# TP 4 — Partie 4 : le temps dans un flux

Dans un flux, deux horloges coexistent :

| Horloge | Définition | Colonne |
|---|---|---|
| **heure de l'événement** (*event time*) | le moment où le colis a réellement changé d'état | `horodatage`, dans le message |
| **heure de traitement** (*processing time*) | le moment où le message arrive dans Kafka ou dans Spark | `recu_le` |

Les deux ne se suivent pas toujours : un scanner hors réseau envoie ses événements plusieurs heures plus tard, et un transporteur renvoie parfois un message déjà envoyé. Les indicateurs métier, comme les incidents d'une journée, se calculent sur l'**heure de l'événement**. Il faut donc décider combien de temps attendre les retardataires.

**Objectif :** construire la couche Silver du flux (événements propres et sans doublons), un indicateur par fenêtre de temps en Gold, et l'état courant de chaque colis.

> **⚠️ Un seul notebook actif à la fois**, et le profil `streaming` démarré.

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F

from flux import (CHECKPOINTS, KAFKA, SCHEMA_EVENEMENT, TOPIC, a_completer, arreter_flux, creer_topic,
                  rejouer, vider_s3)

spark = (SparkSession.builder.appName("tp4-temps")
         .config("spark.ui.showConsoleProgress", "false")   # pas de barres de progression
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")     # n'affiche que les erreurs de Spark
arreter_flux(spark)
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
for t in ["silver.suivi_evenements", "silver.suivi_quarantaine", "silver.colis_etat", "gold.incidents_par_jour"]:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse.{t} PURGE")
for c in ["silver", "quarantaine", "colis_etat", "incidents"]:
    vider_s3(f"{CHECKPOINTS}/{c}")
print("prêt")

## Étape 0 — Remplir Bronze

Pour que cette partie soit indépendante de la précédente, on remplit le topic avec le mois de mars 2018 et on reconstruit `bronze.suivi_evenements` avec la requête de la partie 3, lancée en mode **`availableNow`** : elle traite tout ce qui est disponible, puis s'arrête seule.

In [ ]:
creer_topic(TOPIC)
n_envoyes = rejouer("--depuis", "2018-03-01", "--jusqu-a", "2018-04-01", "--sans-pause")

spark.sql("DROP TABLE IF EXISTS lakehouse.bronze.suivi_evenements PURGE")
vider_s3(f"{CHECKPOINTS}/bronze")
kafka = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
(kafka.select(F.col("key").cast("string").alias("cle"), F.col("value").cast("string").alias("valeur"),
              F.col("partition").alias("kafka_partition"), F.col("offset").alias("kafka_offset"),
              F.col("timestamp").alias("recu_le"))
      .writeStream.trigger(availableNow=True)
      .option("checkpointLocation", f"{CHECKPOINTS}/bronze")
      .toTable("lakehouse.bronze.suivi_evenements")
      .awaitTermination())
print("Bronze :", spark.table("lakehouse.bronze.suivi_evenements").count(), "lignes")

## Étape 1 — Mesurer le désordre

Dans chaque partition, les messages sont rangés par offset, c'est-à-dire dans l'ordre d'arrivée. Pour chaque événement, on regarde l'heure d'événement **la plus récente déjà arrivée avant lui** dans sa partition. S'il est plus ancien qu'elle, il est arrivé **en retard**, et l'écart mesure ce retard.

In [ ]:
evenements_batch = (spark.table("lakehouse.bronze.suivi_evenements")
                    .withColumn("e", F.from_json("valeur", SCHEMA_EVENEMENT))
                    .filter(F.col("e.evenement_id").isNotNull())
                    .withColumn("horodatage_ts", F.to_timestamp("e.horodatage")))
avant_lui = Window.partitionBy("kafka_partition").orderBy("kafka_offset") \
    .rowsBetween(Window.unboundedPreceding, -1)
retards = (evenements_batch
           .withColumn("plus_recent_avant", F.max("horodatage_ts").over(avant_lui))
           .withColumn("retard_h", (F.unix_timestamp("plus_recent_avant") - F.unix_timestamp("horodatage_ts")) / 3600))
(retards.withColumn("tranche", F.when(F.col("retard_h").isNull() | (F.col("retard_h") <= 1),
                                       "1. à l'heure (moins d'1 h)")
                                .when(F.col("retard_h") <= 12, "2. de 1 à 12 h de retard")
                                .when(F.col("retard_h") <= 24, "3. de 12 à 24 h de retard")
                                .otherwise("4. plus de 24 h de retard"))
        .groupBy("tranche").count().orderBy("tranche").show(truncate=False))

> **❓ Question 1.** Quelle part des événements arrive en retard ? Si l'on calculait les incidents d'une journée à minuit pile, avec ce qui est arrivé jusque-là, le résultat serait-il juste ?

## Étape 2 — Silver : quarantaine et doublons

On lit maintenant Bronze **en continu** (`spark.readStream.table`), et on sépare :
- les messages illisibles, qui vont en quarantaine (code fourni) ;
- les événements valides, sans doublons, qui vont dans `silver.suivi_evenements`.

Un transporteur peut renvoyer un message déjà envoyé : le même `evenement_id` arrive alors deux fois. Pour l'écarter, Spark doit se souvenir des identifiants déjà vus… mais pas indéfiniment. Le **watermark** fixe cette limite : `withWatermark("horodatage_ts", "2 days")` signifie qu'on n'attend pas un événement plus de 2 jours après l'heure d'événement la plus récente déjà vue. Au-delà, Spark oublie les identifiants et l'état reste borné.

Complétez `valides` : à partir de `lu`,
1. gardez les lignes dont `e.evenement_id` n'est pas `null`, et sélectionnez `"e.*"` et `recu_le` ;
2. ajoutez `horodatage_ts`, la conversion de `horodatage` en `timestamp` ;
3. déclarez le watermark de 2 jours sur `horodatage_ts` ;
4. retirez les doublons avec `.dropDuplicatesWithinWatermark(["evenement_id"])`.

In [ ]:
lu = (spark.readStream.table("lakehouse.bronze.suivi_evenements")
      .withColumn("e", F.from_json("valeur", SCHEMA_EVENEMENT)))

quarantaine = lu.filter(F.col("e.evenement_id").isNull()).select("valeur", "recu_le")

valides = a_completer("valides")  # À COMPLÉTER

q1 = (quarantaine.writeStream.trigger(availableNow=True)
      .option("checkpointLocation", f"{CHECKPOINTS}/quarantaine")
      .toTable("lakehouse.silver.suivi_quarantaine"))
q2 = (valides.writeStream.trigger(availableNow=True)
      .option("checkpointLocation", f"{CHECKPOINTS}/silver")
      .toTable("lakehouse.silver.suivi_evenements"))
q1.awaitTermination()
q2.awaitTermination()

n_bronze = spark.table("lakehouse.bronze.suivi_evenements").count()
n_quarantaine = spark.table("lakehouse.silver.suivi_quarantaine").count()
n_silver = spark.table("lakehouse.silver.suivi_evenements").count()
print(f"Bronze : {n_bronze} | quarantaine : {n_quarantaine} | Silver : {n_silver} "
      f"| doublons écartés : {n_bronze - n_quarantaine - n_silver}")

> **❓ Question 2.** Combien de doublons ont été écartés ? Pourquoi `dropDuplicates` sans watermark serait-il dangereux sur un flux qui tourne pendant des mois ? Et un watermark de 1 heure, plutôt que 2 jours ?

## Étape 3 — Gold : les incidents par jour et par transporteur

On veut le nombre d'**incidents** par jour et par transporteur, calculé en continu. On regroupe les événements par **fenêtre de temps** : `F.window("horodatage_ts", "1 day")` range chaque événement dans la journée où il s'est produit.

En mode `append`, Spark n'écrit une journée qu'une fois **close** : quand le watermark a dépassé sa fin, plus aucun retardataire n'est attendu, et le chiffre est définitif.

Complétez la fonction `incidents_en_flux(attente)`. Elle lit Kafka par micro-lots de 500 messages (pour simuler l'arrivée progressive), puis :
1. garde les incidents : `categorie = 'incident'` ;
2. déclare le watermark sur `horodatage_ts`, avec la durée `attente` ;
3. compte les incidents par `F.window("horodatage_ts", "1 day")` et par `transporteur`.

Elle renvoie un DataFrame avec les colonnes `jour`, `transporteur` et `incidents`.

In [ ]:
def incidents_en_flux(attente):
    evenements = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", KAFKA)
                  .option("subscribe", TOPIC).option("startingOffsets", "earliest")
                  .option("maxOffsetsPerTrigger", 500).load()
                  .select(F.from_json(F.col("value").cast("string"), SCHEMA_EVENEMENT).alias("e"))
                  .select("e.*")
                  .withColumn("horodatage_ts", F.to_timestamp("horodatage")))
    return a_completer("incidents_en_flux")  # À COMPLÉTER


vider_s3(f"{CHECKPOINTS}/incidents")
(incidents_en_flux("12 hours").writeStream.outputMode("append").trigger(availableNow=True)
    .option("checkpointLocation", f"{CHECKPOINTS}/incidents")
    .toTable("lakehouse.gold.incidents_par_jour")
    .awaitTermination())
spark.table("lakehouse.gold.incidents_par_jour").orderBy("jour", "transporteur").show(8)

Comparons au calcul **batch** sur le même topic, qui voit tous les événements d'un coup, retardataires compris.

In [ ]:
incidents_batch = (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA)
                   .option("subscribe", TOPIC).option("startingOffsets", "earliest")
                   .option("endingOffsets", "latest").load()
                   .select(F.from_json(F.col("value").cast("string"), SCHEMA_EVENEMENT).alias("e"))
                   .select("e.*").filter("categorie = 'incident'")
                   .groupBy(F.date_trunc("day", F.to_timestamp("horodatage")).alias("jour"), "transporteur")
                   .agg(F.count("*").alias("incidents_batch")))


def comparer(table_flux):
    c = (table_flux.join(incidents_batch, ["jour", "transporteur"], "right")
         .withColumn("incidents", F.coalesce("incidents", F.lit(0))))
    jours_clos = table_flux.select("jour").distinct().count()
    totaux = (c.join(table_flux.select("jour").distinct(), "jour")
              .agg(F.sum("incidents").alias("flux"), F.sum("incidents_batch").alias("batch")).first())
    print(f"journées closes : {jours_clos} sur {incidents_batch.select('jour').distinct().count()}")
    print(f"incidents sur ces journées : flux {totaux['flux']}, batch {totaux['batch']}, "
          f"écart {totaux['batch'] - totaux['flux']} ({100 * (totaux['batch'] - totaux['flux']) / totaux['batch']:.1f} %)")


comparer(spark.table("lakehouse.gold.incidents_par_jour"))

Recommençons avec une attente de 2 jours, dans une table en mémoire.

In [ ]:
vider_s3(f"{CHECKPOINTS}/incidents_2j")
(incidents_en_flux("2 days").writeStream.format("memory").queryName("incidents_2j")
    .outputMode("append").trigger(availableNow=True)
    .option("checkpointLocation", f"{CHECKPOINTS}/incidents_2j")
    .start().awaitTermination())
comparer(spark.table("incidents_2j"))

> **❓ Question 3.** Avec 12 heures d'attente, d'où vient l'écart avec le batch ? Que change une attente de 2 jours, sur l'écart et sur le nombre de journées closes ? Pour un tableau de bord consulté chaque matin par le service client, quelle attente choisiriez-vous ?

## Étape 4 — L'état courant de chaque colis

Le service client veut savoir **où en est chaque colis maintenant** : une ligne par colis, avec sa dernière étape. Ce n'est plus un ajout de lignes : il faut **mettre à jour** une table. On utilise `foreachBatch`, qui confie chaque micro-lot à une fonction Python, et un `MERGE` Iceberg.

Complétez le `MERGE` : quand le colis existe déjà, on ne le met à jour **que si** l'événement reçu est plus récent que celui de la table (`s.horodatage_ts > t.horodatage_ts`).

In [ ]:
spark.sql("""CREATE TABLE IF NOT EXISTS lakehouse.silver.colis_etat (colis_id STRING, transporteur STRING,
    code STRING, horodatage_ts TIMESTAMP, ville STRING, etat STRING) USING iceberg""")


def mettre_a_jour(lot, numero):
    plus_recent = Window.partitionBy("colis_id").orderBy(F.col("horodatage_ts").desc())
    (lot.filter("categorie = 'etape'")
        .withColumn("rang", F.row_number().over(plus_recent)).filter("rang = 1")
        .select("colis_id", "transporteur", "code", "horodatage_ts", "ville", "etat")
        .createOrReplaceTempView("maj"))
    # À COMPLÉTER : la condition de mise à jour, à la place de « 1 = 1 »
    lot.sparkSession.sql("""MERGE INTO lakehouse.silver.colis_etat t USING maj s ON t.colis_id = s.colis_id
        WHEN MATCHED AND 1 = 1 THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")


(spark.readStream.table("lakehouse.silver.suivi_evenements")
    .writeStream.foreachBatch(mettre_a_jour).trigger(availableNow=True)
    .option("checkpointLocation", f"{CHECKPOINTS}/colis_etat")
    .start().awaitTermination())

etat = spark.table("lakehouse.silver.colis_etat")
print("colis suivis :", etat.count())
etat.groupBy("code").count().orderBy(F.desc("count")).show()

> **❓ Question 4.** Pourquoi la condition `s.horodatage_ts > t.horodatage_ts` est-elle indispensable ? Donnez un exemple tiré de l'étape 1. Pourquoi ne peut-on pas écrire cette table avec un simple `writeStream` en mode `append` ?

## À retenir

- Les indicateurs se calculent sur l'**heure de l'événement**, pas sur l'heure d'arrivée.
- Le **watermark** dit combien de temps attendre les retardataires : il borne l'état de Spark, et règle le compromis entre fraîcheur et exactitude.
- En mode `append`, une fenêtre n'est écrite qu'une fois close. Pour **mettre à jour** une table, on utilise `foreachBatch` et un `MERGE`.